# Challenge 2: Selecting a constrained portfolio using quantum optimization
### Starter notebook for participants

**Qiskit Fall Fest 2026 · University of Windsor**

**Team:** [team name] · **Members:** [names]

**Your task.** Using a shared synthetic dataset of expected returns and asset covariances, select exactly $K$ assets while balancing expected return against risk. Adapt the supplied Max Cut QAOA workflow to encode and solve this constrained problem.

This notebook has everything you need, in two parts:
* **Part A: Tutorial.** *Combinatorial Optimization with QAOA: Max Cut* by Benjamin Tirado walks through the full QAOA workflow on a small graph. Run it first: the challenge adapts this workflow.
* **Part B: Challenge 2 starter.** The setup, a small practice dataset, the challenge dataset, the problem statement, helper functions for checking and scoring results, and a template for each level. The parts you write yourselves are marked **TODO**.

**Not included:** a portfolio Hamiltonian, a QAOA implementation for the portfolio problem, worked portfolio answers or results. Part A solves Max Cut, a different problem: it is the workflow the challenge asks you to adapt.

| Key dates | |
|---|---|
| Kickoff | Friday, November 13, 3:45 to 4:45 PM |
| Final presentations and judging | Saturday, November 14, starting at 1:00 PM |
| Full schedule | https://qiskit.cs.uwindsor.ca/ |

**Contents**

*Part A: Tutorial, QAOA on Max Cut:* Background · Pre-requisites · Defining the Problem · From Cost Function to Hamiltonian · Building the QAOA Circuit · Optimizing the Parameters · Sampling and Reading the Solution · Results and Evaluation

*Part B: Challenge 2 starter:* 0. Setup · 1. Data · 2. The problem · 3. Toolkit · 4. Level 1: Beginner · 5. Level 2: Intermediate · 6. Level 3: Advanced · 7. What to submit · 8. Resources · 9. Credits

# Part A: Tutorial, QAOA on Max Cut
The tutorial below is reproduced from B. Tirado, *Combinatorial Optimization with QAOA: Max Cut* (BasQ Qiskit Fall Fest, 2026) under CC BY 4.0 (text) and Apache 2.0 (code); see Credits at the end. Run it from top to bottom before starting Part B.

# <b>Combinatorial Optimization with QAOA: Max-Cut </b>
**Author:** Benjamin Tirado  
**Created for:** BasQ Qiskit Fall Fest 2026 — Basque Quantum (BasQ)

<a id="goal"></a>
<div class="alert alert-block alert-success">
    
<b>Goal of this notebook: </b> Solve the **Max-Cut problem** on a small graph using the **Quantum Approximate Optimization Algorithm (QAOA)**. We will translate a combinatorial problem into a **cost Hamiltonian**, build the QAOA circuit from alternating **cost** and **mixer** layers, optimize its parameters with a classical routine, and sample the result to read out a good partition — checking it against a brute-force solution. By the end you will have a reusable QAOA template that you can point at harder, more realistic optimization problems.
</div>

## <b>Background </b> <a id="background"></a>

Many important problems in logistics, finance, scheduling, and network design are **combinatorial optimization** problems: choose the best option out of an enormous, discrete set of possibilities. Because the number of candidate solutions grows exponentially with problem size, exact classical methods struggle as problems get large, and much of computer science is devoted to good approximate methods.

Quantum computers offer a different angle of attack. The **Quantum Approximate Optimization Algorithm (QAOA)**, introduced by Farhi, Goldstone and Gutmann in 2014, is a hybrid quantum-classical algorithm designed to find good approximate solutions to such problems. It has since become one of the most studied quantum optimization methods, and a decade later it sits at the heart of utility-scale demonstrations on IBM hardware (for example the error-suppressed pipeline of Sachdeva et al., 2024, which tackled nontrivial binary optimization at the 100+ qubit scale).

The idea behind QAOA is elegant. Any problem where we want to *minimize a cost* can be encoded into a **cost Hamiltonian** $\hat{H}_C$, built so that its lowest-energy state corresponds to the best solution. QAOA prepares a trial state by alternating two ingredients:

$$
|\psi(\vec{\gamma}, \vec{\beta})\rangle = \prod_{k=1}^{p} e^{-i\beta_k \hat{H}_M}\, e^{-i\gamma_k \hat{H}_C}\; |+\rangle^{\otimes n},
$$

where $\hat{H}_C$ is the **cost** Hamiltonian (it rewards good solutions), $\hat{H}_M = \sum_i X_i$ is the **mixer** (it explores different candidate solutions), and the integer $p$ is the number of **layers** (also called the QAOA depth). The angles $\vec{\gamma}$ and $\vec{\beta}$ are tuned by a classical optimizer to minimize the expected cost. As $p$ grows, QAOA can represent better and better solutions, at the price of deeper circuits.

As our teaching problem we use **Max-Cut**, the standard entry point to QAOA: it maps onto a cost Hamiltonian in the simplest possible way, and on small graphs we can check the quantum answer against brute force.

# <b>Pre-requisites </b> <a id="prereq"></a>
For starters, make sure you have installed the Qiskit SDK and its supporting modules: `qiskit_aer` (for noiseless simulations) and `qiskit_ibm_runtime` (for real-hardware experiments), as well as the visualization package `qiskit[visualization]`. We use `qiskit.circuit.library.QAOAAnsatz` to build the QAOA circuit, the built-in `StatevectorEstimator` and `StatevectorSampler` primitives to evaluate and sample it, and `scipy.optimize` for the classical parameter optimization. Graphs are handled with `rustworkx`, Qiskit's companion graph library.

In [ ]:
# %pip install qiskit qiskit_aer qiskit_ibm_runtime qiskit[visualization] rustworkx

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
from itertools import product

from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from scipy.optimize import minimize

# <b>Defining the Problem: Max-Cut on a Graph </b> <a id="problem"></a>
The **Max-Cut** problem is easy to state. Given a graph of nodes connected by edges, split the nodes into **two groups** so that the number of edges running *between* the groups (the edges we "cut") is as large as possible.

We describe a partition by assigning each node a label $x_i \in \{0, 1\}$ — which side of the cut it is on. An edge $(i, j)$ is cut exactly when its two endpoints have different labels. The number of cut edges is therefore

$$
C(\vec{x}) = \sum_{(i,j)\in E} \big[\, x_i \ne x_j \,\big],
$$

and Max-Cut asks us to maximize this over all $2^n$ possible labellings. Let us build a small graph to work with — five nodes, chosen so we can still check every possibility by hand.

In [ ]:
# Build a small 5-node graph
num_nodes = 5
edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 4)]

graph = rx.PyGraph()
graph.add_nodes_from(range(num_nodes))
graph.add_edges_from([(i, j, 1.0) for i, j in edges])  # weight 1 on every edge

draw_graph(graph, node_size=600, with_labels=True, node_color="#9ecae1")

# <b>From Cost Function to Hamiltonian </b> <a id="hamiltonian"></a>
To run Max-Cut on a quantum computer we translate the cost function into a **cost Hamiltonian** whose ground state encodes the optimal cut. The standard trick is to replace each binary variable $x_i \in \{0,1\}$ by a spin $Z_i \in \{+1, -1\}$ via $x_i = (1 - Z_i)/2$. Substituting this into the Max-Cut objective, each edge $(i,j)$ contributes

$$
\big[\, x_i \ne x_j \,\big] \;\longrightarrow\; \tfrac{1}{2}\big(1 - Z_i Z_j\big).
$$

The term $Z_i Z_j$ equals $+1$ when both endpoints are on the same side (edge not cut) and $-1$ when they differ (edge cut). Because quantum systems naturally settle into **minimum** energy, and we want to **maximize** the cut, we minimize the following cost Hamiltonian, dropping constant offsets:

$$
\hat{H}_C = \sum_{(i,j)\in E} \tfrac{1}{2}\, Z_i Z_j.
$$

Its ground state is the spin configuration — the partition — that cuts the most edges. We build it below as a `SparsePauliOp`, one Pauli-`ZZ` string per edge.

In [ ]:
def maxcut_cost_hamiltonian(num_nodes, edges):
    """Build the Max-Cut cost Hamiltonian sum_(i,j) 0.5 * Z_i Z_j."""
    terms = []
    for i, j in edges:
        pauli = ["I"] * num_nodes
        pauli[i] = "Z"
        pauli[j] = "Z"
        # Qiskit reads Pauli strings right-to-left (qubit 0 is the rightmost char)
        terms.append(("".join(reversed(pauli)), 0.5))
    return SparsePauliOp.from_list(terms)


cost_hamiltonian = maxcut_cost_hamiltonian(num_nodes, edges)
print("Cost Hamiltonian (one ZZ term per edge):")
print(cost_hamiltonian)

# <b>Building the QAOA Circuit </b> <a id="circuit"></a>
With the cost Hamiltonian in hand, Qiskit can build the QAOA circuit for us. The `QAOAAnsatz` takes the cost operator and a number of repetitions `reps` (this is the depth $p$) and assembles the alternating pattern automatically:

- it starts every qubit in the equal superposition $|+\rangle$ (all candidate solutions at once),
- applies the **cost layer** $e^{-i\gamma_k \hat{H}_C}$ (which imprints the problem structure), and
- applies the **mixer layer** $e^{-i\beta_k \hat{H}_M}$ with $\hat{H}_M = \sum_i X_i$ (which shuffles amplitude between candidates),

repeating for $p$ layers. This leaves $2p$ free parameters (one $\gamma$ and one $\beta$ per layer) for the optimizer to tune.

We start with the shallowest non-trivial case, $p = 1$, which has just two parameters. We add measurements so we can later sample bitstrings from the circuit.

In [ ]:
p = 1  # QAOA depth (number of layers)
ansatz = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=p)
ansatz.measure_all()

print(f"QAOA depth p = {p}")
print("Number of qubits:", ansatz.num_qubits)
print("Number of parameters:", ansatz.num_parameters)
ansatz.decompose(reps=3).draw("mpl", fold=-1)

# <b>Optimizing the Parameters </b> <a id="optimize"></a>
QAOA is a **hybrid** algorithm: the quantum computer prepares the trial state and estimates its energy, while a **classical optimizer** adjusts the angles $(\vec{\gamma}, \vec{\beta})$ to drive that energy down. The quantity we minimize is the expected value of the cost Hamiltonian,

$$
E(\vec{\gamma}, \vec{\beta}) = \langle \psi(\vec{\gamma}, \vec{\beta}) | \hat{H}_C | \psi(\vec{\gamma}, \vec{\beta}) \rangle,
$$

which we evaluate with the `StatevectorEstimator` (exact and noiseless — ideal for learning). We wrap this in a small cost function and hand it to `scipy.optimize.minimize` with the **COBYLA** optimizer, a gradient-free method that copes well with the noisy, non-convex landscapes typical of QAOA.

Note we pass the ansatz *without* its measurements to the Estimator (expectation values are computed from the statevector, not from samples); the measured copy is kept for the sampling step that follows.

In [ ]:
estimator = StatevectorEstimator()

# Estimator needs the circuit without measurements
ansatz_no_meas = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=p)

objective_history = []

def cost_function(params):
    """Expected energy of the cost Hamiltonian for given QAOA angles."""
    pub = (ansatz_no_meas, cost_hamiltonian, params)
    energy = estimator.run([pub]).result()[0].data.evs
    energy = float(energy)
    objective_history.append(energy)
    return energy

# A reasonable starting point; QAOA is sensitive to initialization.
np.random.seed(42)
initial_params = np.random.uniform(0, np.pi, ansatz_no_meas.num_parameters)

result = minimize(cost_function, initial_params, method="COBYLA",
                  options={"maxiter": 200})

print("Optimization finished.")
print("Optimal parameters:", np.round(result.x, 4))
print(f"Minimum expected energy: {result.fun:.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(objective_history, color="#1f77b4")
plt.xlabel("Optimizer iteration")
plt.ylabel(r"Expected cost  $\langle H_C \rangle$")
plt.title("QAOA optimization convergence")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# <b>Sampling and Reading the Solution </b> <a id="sample"></a>
Minimizing the energy gives us good angles, but the *solution* we care about is a **bitstring** — an actual assignment of nodes to the two sides of the cut. To get it, we bind the optimal parameters into the measured circuit and **sample** it many times with the `StatevectorSampler`. The most frequently observed bitstrings are QAOA's candidate solutions.

For each sampled bitstring we compute how many edges it cuts, and keep the best. On a small graph the optimal partition should dominate the sampled distribution when QAOA has done its job.

In [ ]:
def cut_value(bitstring, edges):
    """Number of edges cut by a given assignment (bitstring)."""
    # bitstring[i] is the side (0 or 1) assigned to node i
    return sum(1 for i, j in edges if bitstring[i] != bitstring[j])


sampler = StatevectorSampler()
best = ansatz.assign_parameters(result.x)
samples = sampler.run([best], shots=4096).result()[0].data.meas.get_counts()

# Convert each measured key (a bit-string) into a per-node assignment.
# Qiskit bitstrings are little-endian: the rightmost char is qubit 0.
def key_to_assignment(key, num_nodes):
    bits = key.replace(" ", "")[::-1]  # reverse -> index 0 is node 0
    return [int(bits[i]) for i in range(num_nodes)]

scored = []
for key, count in samples.items():
    assignment = key_to_assignment(key, num_nodes)
    scored.append((assignment, count, cut_value(assignment, edges)))

# Sort by how often each bitstring appeared
scored.sort(key=lambda t: t[1], reverse=True)
print("Most sampled bitstrings (assignment, count, cut value):")
for assignment, count, cut in scored[:5]:
    print(f"  {assignment}   count={count:5d}   cut={cut}")

# <b>Results and Evaluation </b> <a id="res"></a>
Because the graph is small, we can **brute-force** the true optimum: enumerate all $2^n$ partitions, compute each cut value, and find the maximum. Comparing QAOA's best sampled cut to this exact optimum tells us how well the algorithm did.

A common summary metric is the **approximation ratio**, the ratio of the cut QAOA found to the true optimal cut. A ratio of $1.0$ means QAOA recovered an optimal partition. At $p = 1$ on a simple graph you should already get close; one of the first things you will explore in the challenge is how this ratio improves as the depth $p$ grows.

In [ ]:
# Brute-force optimum
best_cut, best_assignments = 0, []
for bits in product([0, 1], repeat=num_nodes):
    c = cut_value(bits, edges)
    if c > best_cut:
        best_cut, best_assignments = c, [bits]
    elif c == best_cut:
        best_assignments.append(bits)

qaoa_best = max(scored, key=lambda t: t[2])
qaoa_cut = qaoa_best[2]
approx_ratio = qaoa_cut / best_cut

print(f"Brute-force optimal cut : {best_cut}")
print(f"QAOA best sampled cut   : {qaoa_cut}")
print(f"Approximation ratio     : {approx_ratio:.3f}")

# Visualize the QAOA partition
colors = ["#f4a582" if b == 0 else "#92c5de" for b in qaoa_best[0]]
draw_graph(graph, node_size=600, with_labels=True, node_color=colors)

# Part B: Challenge 2 starter

## 0. Setup
The Qiskit packages are installed in Part A. Run the imports below; the constants are part of your computational budget, so report the values you used.

In [ ]:
# %pip install scipy matplotlib

In [ ]:
%matplotlib inline
import json
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

import qiskit, qiskit_aer
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeTorino

SEED = 2026            # fix every random choice so your results can be rerun
RISK_AVERSION = 2.0    # q in f(x) = q x^T Sigma x - mu^T x; use the value supplied with the shared dataset if one is given
SHOTS = 4096           # shots per sampling run

print(f"qiskit {qiskit.__version__} | qiskit_aer {qiskit_aer.__version__} | numpy {np.__version__}")

## 1. Data
### 1.1 Practice data (5 assets, for understanding only)
A small, readable instance for trying out your code and checking it by hand: five assets, hold $K = 2$, risk weight $q = 1$. It has $\binom{5}{2} = 10$ valid portfolios out of $2^5 = 32$ bitstrings. **Do not submit results on this data:** use the challenge dataset in 1.2.

In [ ]:
asset_names = ["Asset A", "Asset B", "Asset C", "Asset D", "Asset E"]
mu = np.array([0.12, 0.10, 0.08, 0.09, 0.07], dtype=float)   # expected returns
Sigma = np.array([                                            # covariance matrix
    [0.040, 0.012, 0.004, 0.006, 0.008],
    [0.012, 0.025, 0.003, 0.005, 0.006],
    [0.004, 0.003, 0.016, 0.002, 0.004],
    [0.006, 0.005, 0.002, 0.032, 0.009],
    [0.008, 0.006, 0.004, 0.009, 0.024],
], dtype=float)
K = 2              # hold exactly K assets
risk_weight = 1.0  # q in f(x) = q x^T Sigma x - mu^T x

n_assets = len(mu)
assert Sigma.shape == (n_assets, n_assets)
assert len(asset_names) == n_assets
assert 0 <= K <= n_assets
assert np.allclose(Sigma, Sigma.T)
assert np.linalg.eigvalsh(Sigma).min() >= -1e-10

print(f"{'asset':<9}{'return':>8}{'volatility':>12}")
for name, m, v in zip(asset_names, mu, np.sqrt(np.diag(Sigma))):
    print(f"{name:<9}{m:8.3f}{v:12.3f}")

### 1.2 Challenge dataset (16 assets)
**Use this dataset for all results you submit, at every level.** If the organizers release an official dataset file, set `DATA_PATH` to it and run the cell. The loader accepts a `.npz` or `.json` file holding an expected return vector and a covariance matrix (common key names are recognised), or a pair of `.csv` files given as `("returns.csv", "covariance.csv")`.

While `DATA_PATH = None`, the cell builds a seeded 16 asset dataset (covariance from a three factor model, symmetric and positive definite). The seed is fixed, so every team gets exactly the same numbers. Its risk weight is `RISK_AVERSION` from the setup cell.

In [ ]:
DATA_PATH = None   # e.g. "portfolio_dataset.npz" or ("returns.csv", "covariance.csv")

def make_standin(n_assets=16, n_factors=3, seed=SEED):
    rng = np.random.default_rng(seed)
    vol = rng.uniform(0.10, 0.35, n_assets)
    L = rng.normal(size=(n_assets, n_factors))
    L /= np.linalg.norm(L, axis=1, keepdims=True)
    share = rng.uniform(0.2, 0.6, n_assets)            # share of variance explained by the factors
    L *= np.sqrt(share)[:, None]
    corr = L @ L.T + np.diag(1 - share)
    Sigma = np.outer(vol, vol) * corr
    mu = 0.02 + vol * rng.uniform(0.1, 0.6, n_assets)  # return grows with risk, with noise
    return mu, Sigma

def load_dataset(path):
    MU_KEYS = ("mu", "returns", "expected_returns", "mean_returns", "r")
    SIG_KEYS = ("Sigma", "sigma", "cov", "covariance", "covariances", "S")
    if isinstance(path, (tuple, list)):
        mu = np.loadtxt(path[0], delimiter=",", ndmin=1)
        Sigma = np.loadtxt(path[1], delimiter=",", ndmin=2)
    elif str(path).endswith(".npz"):
        d = np.load(path)
        mu = d[next(k for k in MU_KEYS if k in d)]; Sigma = d[next(k for k in SIG_KEYS if k in d)]
    elif str(path).endswith(".json"):
        d = json.load(open(path))
        mu = np.array(d[next(k for k in MU_KEYS if k in d)]); Sigma = np.array(d[next(k for k in SIG_KEYS if k in d)])
    else:
        raise ValueError("Unsupported dataset format: use .npz, .json or (returns.csv, covariance.csv)")
    return np.asarray(mu, float).ravel(), np.asarray(Sigma, float)

MU_ALL, SIGMA_ALL = make_standin() if DATA_PATH is None else load_dataset(DATA_PATH)
N_ALL = len(MU_ALL)
assert SIGMA_ALL.shape == (N_ALL, N_ALL) and np.allclose(SIGMA_ALL, SIGMA_ALL.T), "covariance must be square and symmetric"

vols = np.sqrt(np.diag(SIGMA_ALL))
print(f"{'seeded dataset' if DATA_PATH is None else DATA_PATH}: {N_ALL} assets")
print("asset   " + " ".join(f"{i:>6d}" for i in range(N_ALL)))
print("return  " + " ".join(f"{m:6.3f}" for m in MU_ALL))
print("vol     " + " ".join(f"{v:6.3f}" for v in vols))

## 2. The problem
**Decision.** $x_i \in \{0, 1\}$, where 1 means asset $i$ is held. Qubit $i$ stands for asset $i$.

**Objective** (lower is better): minimize $\;f(x) = q\,x^\top \Sigma x - \mu^\top x$, that is risk minus return. The risk aversion $q$ sets the trade off. This is an unnormalized selection score: each held asset counts once. If you prefer equally weighted portfolios, scale the return by $1/K$ and the risk by $1/K^2$, say so, and use it consistently.

**Notation:** $q$ is the risk weight and $\lambda$ is the penalty strength, as on the challenge slides.

**Constraint:** $x_1 + x_2 + \dots + x_n = K$. A bitstring that satisfies it is called **valid**.

**To use QAOA you must:**
1. Fold the constraint in: a penalty $\lambda\left(\sum_i x_i - K\right)^2$, or a circuit that never breaks it.
2. Turn bits into spins with $x_i = (1 - Z_i)/2$, as the tutorial does for Max Cut.
3. Collect the $Z_i$ and $Z_iZ_j$ terms into a cost Hamiltonian, and check it against brute force.

**Reporting rule.** Report valid solutions (exactly $K$ assets) separately from penalized objective values.

## 3. Toolkit
These helpers are provided so that every team checks and scores results the same way. They do not solve the challenge: the encoding, the quantum algorithm and every setting are yours to choose.

| Helper | What it does |
|---|---|
| `portfolio_cost(x, mu, Sigma)` | $f(x)$ for a 0/1 vector $x$ |
| `key_to_bits(key)` | Qiskit measurement key to a vector with `x[i]` = qubit $i$ (Qiskit prints qubit 0 on the **right**) |
| `brute_force(mu, Sigma, K, q)` | exact reference: best valid portfolio $f^\star$, worst valid cost $f_{\max}$, number of valid portfolios |
| `evaluate_counts(counts, mu, Sigma, K, ref)` | the challenge metrics from sampled counts |
| `check_hamiltonian(H, offset, target)` | compares $\langle x \vert H \vert x\rangle$ + offset with any classical cost on every bitstring |
| `circuit_resources(qc, backend)` | qubits, depth and two qubit gate count after transpiling |

In [ ]:
def portfolio_cost(x, mu, Sigma, q=RISK_AVERSION):
    """Objective f(x) = q x^T Sigma x - mu^T x for a 0/1 vector x (lower is better)."""
    x = np.asarray(x, float)
    return float(q * x @ Sigma @ x - mu @ x)

def key_to_bits(key):
    """Qiskit measurement key -> array x with x[i] = bit of qubit i.
    Qiskit prints qubit 0 as the RIGHTMOST character, so the key is read backwards."""
    return np.array([int(c) for c in key.replace(" ", "")[::-1]], dtype=int)

def bits_to_key(x):
    """Inverse of key_to_bits: vector with x[i] = qubit i -> Qiskit style key."""
    return "".join(str(int(b)) for b in x[::-1])

def bits_to_assets(x):
    return [i for i, b in enumerate(x) if b]

def brute_force(mu, Sigma, K, q=RISK_AVERSION):
    """Exact reference by scoring every portfolio with exactly K assets (fine up to about 20 assets)."""
    n = len(mu)
    ranking = []
    for S in combinations(range(n), K):
        x = np.zeros(n); x[list(S)] = 1
        ranking.append((portfolio_cost(x, mu, Sigma, q), list(S)))
    ranking.sort(key=lambda t: t[0])
    return dict(f_opt=ranking[0][0], best_assets=ranking[0][1], f_max=ranking[-1][0],
                n_valid=len(ranking), ranking=ranking)

def evaluate_counts(counts, mu, Sigma, K, ref, q=RISK_AVERSION):
    """Challenge metrics from a Qiskit counts dict.
    valid       share of shots that hold exactly K assets
    p_best      share of shots that are an optimal portfolio
    quality     (f_max - mean f of valid shots) / (f_max - f_opt): 1 is perfect, a random valid portfolio scores about 0.5
    best_assets best valid portfolio observed, and gap = f(best observed) - f_opt
    """
    shots = sum(counts.values())
    n_valid = n_best = 0
    f_sum, best = 0.0, None
    for key, c in counts.items():
        x = key_to_bits(key)
        if x.sum() != K:
            continue
        f = portfolio_cost(x, mu, Sigma, q)
        n_valid += c
        f_sum += c * f
        n_best += c * bool(np.isclose(f, ref["f_opt"]))
        if best is None or f < best[0]:
            best = (f, bits_to_assets(x))
    span = ref["f_max"] - ref["f_opt"]
    quality = (ref["f_max"] - f_sum / n_valid) / span if (n_valid and span > 0) else float("nan")
    return dict(valid=n_valid / shots, p_best=n_best / shots, quality=quality,
                best_assets=best[1] if best else None,
                gap=(best[0] - ref["f_opt"]) if best else None)

def diagonal_energies(H, offset=0.0):
    """<x|H|x> + offset for every basis state of an operator made only of I and Z terms.
    Row b of X is the bitstring whose bit i is qubit i."""
    n = H.num_qubits
    X = (np.arange(2**n)[:, None] >> np.arange(n)) & 1
    Zval = 1 - 2 * X                       # Z eigenvalue: +1 for bit 0, -1 for bit 1
    E = np.full(2**n, float(offset))
    for label, c in H.to_list():
        lab = label[::-1]                  # lab[i] acts on qubit i
        if set(lab) - {"I", "Z"}:
            raise ValueError(f"term {label} is not diagonal: use only I and Z")
        idx = [i for i, ch in enumerate(lab) if ch == "Z"]
        E += np.real(c) * (np.prod(Zval[:, idx], axis=1) if idx else 1.0)
    return E, X

def check_hamiltonian(H, offset, target, tol=1e-9):
    """Compare <x|H|x> + offset with target(x) on all 2^n bitstrings (keep n at 14 or below)."""
    E, X = diagonal_energies(H, offset)
    T = np.array([target(x) for x in X])
    err = float(np.max(np.abs(E - T)))
    print(f"largest difference over {len(E)} bitstrings: {err:.2e} -> " +
          ("match" if err < tol else "MISMATCH: check signs, factors of 1/2, the qubit order and the offset"))
    return err < tol

def circuit_resources(qc, backend=None, seed=SEED):
    """Transpile qc for a fully connected device (backend=None) or for a given backend, and report its cost."""
    if backend is None:
        tqc = transpile(qc, basis_gates=["rz", "sx", "x", "cx"], optimization_level=3, seed_transpiler=seed)
    else:
        tqc = transpile(qc, backend=backend, optimization_level=3, seed_transpiler=seed)
    return dict(qubits=qc.num_qubits, depth=tqc.depth(), two_qubit_gates=tqc.num_nonlocal_gates())

### Toolkit examples
**Random guessing.** The baseline every method must beat: uniformly random bitstrings scored with `evaluate_counts`, shown here on the practice data.

In [ ]:
ref_s = brute_force(mu, Sigma, K, q=risk_weight)

rng = np.random.default_rng(SEED)
random_counts = {}
for _ in range(SHOTS):
    key = bits_to_key(rng.integers(0, 2, n_assets))
    random_counts[key] = random_counts.get(key, 0) + 1
print(f"{ref_s['n_valid']} valid portfolios out of {2**n_assets} bitstrings")
print("random guessing:", {k: (round(v, 3) if isinstance(v, float) else v)
                           for k, v in evaluate_counts(random_counts, mu, Sigma, K, ref_s, q=risk_weight).items()})

**Checking a Hamiltonian.** A toy example, unrelated to the portfolio objective: the number of held assets, $\sum_i x_i$. With $x_i = (1 - Z_i)/2$ it becomes $\frac{n}{2} - \frac{1}{2}\sum_i Z_i$, so the operator is $-\frac{1}{2}\sum_i Z_i$ with offset $n/2$. Use the same check on your own Hamiltonian in Level 1.

In [ ]:
H_toy = SparsePauliOp.from_sparse_list([("Z", [i], -0.5) for i in range(n_assets)], num_qubits=n_assets)
check_hamiltonian(H_toy, n_assets / 2, lambda x: x.sum())

**Circuit cost.** The same circuit transpiled for a fully connected device and for `FakeTorino`, a model of a 133 qubit IBM Heron processor with heavy hex connectivity. The example circuit is a generic layer of two qubit gates on every pair.

In [ ]:
qc_demo = QuantumCircuit(n_assets)
qc_demo.h(range(n_assets))
for i, j in combinations(range(n_assets), 2):
    qc_demo.rzz(0.1, i, j)
print("fully connected:", circuit_resources(qc_demo))
print("FakeTorino     :", circuit_resources(qc_demo, FakeTorino()))

## 4. Level 1: Beginner (getting started)
Set up a small portfolio selection instance, encode it as a cost Hamiltonian (objective **plus** a budget constraint penalty), and solve it with QAOA following the workflow from Part A. Then study the two knobs that matter most at this level: how the **approximation ratio improves with QAOA depth $p$**, and how the **penalty strength** on the constraint affects whether the solutions QAOA returns are actually feasible. Verify against a brute force optimum.

### 4.1 Instance and exact reference

In [ ]:
# Submitted results use the challenge dataset. To try your code first, you may temporarily use the practice data:
# mu1, Sigma1, K1, q1 = mu, Sigma, K, risk_weight
N1, K1, q1 = 6, 3, RISK_AVERSION     # a small slice of the challenge dataset that brute force can check
mu1, Sigma1 = MU_ALL[:N1], SIGMA_ALL[:N1, :N1]
ref1 = brute_force(mu1, Sigma1, K1, q=q1)
print(f"n = {N1}, K = {K1}: {ref1['n_valid']} valid portfolios out of {2**N1} bitstrings")

### 4.2 Encode: the cost Hamiltonian (TODO)

In [ ]:
def build_cost_hamiltonian(mu, Sigma, K, lam, q=RISK_AVERSION):
    """Return (H, offset): a SparsePauliOp of Z and ZZ terms and a constant, such that
    <x|H|x> + offset = f(x) + lam * (sum(x) - K)^2 for every bitstring x."""
    # TODO: write your encoding here
    raise NotImplementedError("build_cost_hamiltonian")

LAM = 1.0   # TODO: choose a penalty strength, then study it in 4.4
try:
    H1, offset1 = build_cost_hamiltonian(mu1, Sigma1, K1, LAM, q=q1)
    check_hamiltonian(H1, offset1, lambda x: portfolio_cost(x, mu1, Sigma1, q1) + LAM * (x.sum() - K1) ** 2)
except NotImplementedError as e:
    print("TODO:", e)

### 4.3 Solve: QAOA (TODO)
Follow the workflow from Part A: `QAOAAnsatz`, `StatevectorEstimator` and COBYLA (`scipy.optimize.minimize`), then `StatevectorSampler`. Two Qiskit details:
* `QAOAAnsatz` orders its parameters with all β before all γ. Check `ansatz.parameters`.
* Qiskit 2.1 and later warn that `QAOAAnsatz` is deprecated; the function `qiskit.circuit.library.qaoa_ansatz` is its replacement. Either works for this challenge.

**Questions to address:** Which quantity does your optimizer minimize, and how does it differ from the quality of the portfolios you sample? How do parameter initialization, depth $p$ and the optimization budget change the result?

In [ ]:
def run_qaoa(H, offset, p, seed=SEED):
    """Optimize a depth p QAOA circuit for H. Return (best_params, best_energy, ansatz)."""
    # TODO
    raise NotImplementedError("run_qaoa")

def sample_qaoa(ansatz, params, shots=SHOTS, seed=SEED):
    """Sample the optimized circuit. Return a Qiskit counts dict."""
    # TODO
    raise NotImplementedError("sample_qaoa")

### 4.4 Evaluate: depth $p$ and penalty strength $\lambda$ (TODO)
Fill in the values to study. For each run, record the metrics from `evaluate_counts` and, separately, the penalized energy the optimizer reached.

In [ ]:
P_VALUES = []     # TODO: depths to study
LAM_VALUES = []   # TODO: penalty strengths to study

results_level1 = []
try:
    for p in P_VALUES:
        for lam in LAM_VALUES:
            H, offset = build_cost_hamiltonian(mu1, Sigma1, K1, lam, q=q1)
            params, energy, ansatz = run_qaoa(H, offset, p)
            counts = sample_qaoa(ansatz, params)
            results_level1.append(dict(p=p, lam=lam, penalized_energy=energy,
                                       **evaluate_counts(counts, mu1, Sigma1, K1, ref1, q=q1)))
except NotImplementedError as e:
    print("TODO:", e)

for row in results_level1:
    print(row)
if not results_level1:
    print("No results yet: fill in P_VALUES, LAM_VALUES and the TODO functions.")
# TODO: plot the approximation ratio against p, and the valid fraction against lam

### 4.5 Verify against the brute force optimum (TODO)
Show your best valid sample next to `ref1["best_assets"]` and `ref1["f_opt"]`.

In [ ]:
print("brute force optimum:", ref1["best_assets"], f"f* = {ref1['f_opt']:.5f}")
# TODO: print your best valid sample and its gap

## 5. Level 2: Intermediate (choose your approach)
Scale to a larger or more tightly constrained instance and commit to a strategy. This level **forks**, and part of the task is to justify which path you take and compare it against a sensible classical baseline:

* **Variational / QAOA route.** Push QAOA harder: warm starting from a classical relaxation, constraint preserving or custom mixers that keep the search inside the feasible set, better classical optimizers, or smarter parameter initialization.
* **Quantum machine learning route.** Treat the problem through a learning lens: for example, a variational model trained to produce high quality (feasible) solutions, or a QML approach to the underlying selection/classification structure.

In [ ]:
N2, K2 = 12, 4       # TODO: choose your larger or tighter instance
mu2, Sigma2 = MU_ALL[:N2], SIGMA_ALL[:N2, :N2]
ref2 = brute_force(mu2, Sigma2, K2)
print(f"n = {N2}, K = {K2}: {ref2['n_valid']} valid portfolios out of {2**N2} bitstrings")

# TODO: your chosen route
# TODO: standard QAOA on the same instance, for comparison
# TODO: a sensible classical baseline, run under a comparable budget

## 6. Level 3: Advanced (push toward utility scale)
Move toward the regime where naive encodings and shallow circuits stop working. Explore what it takes to get good solutions as the problem grows: confronting **trainability** issues (barren plateaus), **higher order terms or heavier constraints**, and the **error suppression and hardware aware compilation** that utility scale optimization demands. The inspiration here is the integrated, error suppressed pipelines that have pushed gate model optimization to the 100+ qubit scale: how close can you get to a robust, scalable solver?

**Running on real hardware.** Whichever level you tackle, moving from the `StatevectorEstimator` to a real IBM Quantum device changes the game. The two qubit cost terms must be mapped onto the heavy hex connectivity, and terms between non adjacent qubits force the transpiler to insert SWAPs that deepen the circuit; SWAP networks and commuting gate routing become valuable. Finite shots and gate noise also blur the sampled distribution, so error suppression and mitigation (dynamical decoupling, twirling, and more) become part of getting a trustworthy answer.

In [ ]:
noisy_backend = AerSimulator.from_backend(FakeTorino())   # noise model of a 133 qubit IBM Heron device

SIZES = []           # TODO: problem sizes to study
BUDGET = dict(shots=SHOTS, optimizer="COBYLA", maxiter=None, restarts=None)   # TODO: state your budget

# TODO: how quality, validity and cost change with size under BUDGET
# TODO: trainability, heavier constraints, routing (circuit_resources) and noise (noisy_backend)

## 7. What to submit
**Every team submits**
1. **This notebook, completed and reproducible:** data source, code, package versions, random seeds and how to run it. State the instance ($n$, $K$, $q$), the encoding and the method.
2. **Results with a reference:** the metrics from `evaluate_counts` for every method you try, the portfolios actually selected, and the gap to brute force or a classical baseline, all on the challenge dataset.
3. **A short presentation** for the final presentations and judging on Saturday, November 14, starting at 1:00 PM.

**Checklist**
- [ ] Formulation: objective, exactly $K$ constraint, penalty or mixer
- [ ] Valid fraction and the portfolios actually selected
- [ ] Best valid objective and the gap to the exact or classical reference
- [ ] Resources: qubits, depth, two qubit gates, shots, optimizer evaluations
- [ ] How your method differs from the Max Cut workflow in Part A
- [ ] Limitations

**Rules for every level**
* Report results on the challenge dataset, not the practice data.
* Report valid portfolios separately from penalized energy values: the best penalized energy is not the same as the best valid portfolio.
* Verify against brute force wherever the size allows.
* State your computational budget: shots, optimizer iterations, restarts.
* Make it reproducible: fixed seeds and package versions.

**Results table** (one row per method and setting)

| Method | $n$ | $K$ | $q$ | $\lambda$ | $p$ | Seed | Shots | Valid fraction | P(best) | Quality | Best valid $f$ | $f^\star$ | Gap | Qubits | Depth | Two qubit gates | Optimizer evaluations |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| | | | | | | | | | | | | | | | | | |

## 8. Resources
**QAOA and quantum optimization in Qiskit**
* [Quantum approximate optimization algorithm (IBM tutorial)](https://quantum.cloud.ibm.com/docs/tutorials/quantum-approximate-optimization-algorithm)
* [Advanced techniques for QAOA](https://quantum.cloud.ibm.com/docs/tutorials/advanced-techniques-for-qaoa)
* [QAOAAnsatz documentation](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.QAOAAnsatz)
* [Qiskit Optimization documentation](https://qiskit-community.github.io/qiskit-optimization/)
* [Max Cut and TSP tutorial](https://qiskit-community.github.io/qiskit-optimization/tutorials/06_examples_max_cut_and_tsp.html)
* [Portfolio optimization tutorial](https://qiskit-community.github.io/qiskit-finance/tutorials/01_portfolio_optimization.html)

**Encoding, constraints and optimizers**
* [Converters for quadratic programs](https://qiskit-community.github.io/qiskit-optimization/tutorials/02_converters_for_quadratic_programs.html)
* [Warm starting QAOA](https://qiskit-community.github.io/qiskit-optimization/tutorials/10_warm_start_qaoa.html)
* [SamplerV2 and primitives](https://quantum.cloud.ibm.com/docs/guides/primitives)

**Hardware execution, transpilation and error mitigation**
* [Transpile with pass managers](https://quantum.cloud.ibm.com/docs/guides/transpile-with-pass-managers)
* [Error mitigation and suppression techniques](https://quantum.cloud.ibm.com/docs/guides/error-mitigation-and-suppression-techniques)

**Background reading**
* [Farhi, Goldstone and Gutmann, *A Quantum Approximate Optimization Algorithm* (2014)](https://arxiv.org/abs/1411.4028)
* [Sachdeva et al., *Quantum optimization ... on a 127 qubit gate model IBM quantum computer* (2024)](https://arxiv.org/abs/2406.01743)

## 9. Credits
Prepared by Dr. Shaon Bhatta Shuvo, School of Computer Science, University of Windsor (Challenge 2 mentor), for Qiskit Fall Fest 2026.

Part A, the challenge text for each level, the hardware paragraph and the resource list come from B. Tirado, *Combinatorial Optimization with QAOA: Max Cut*, tutorial notebook, BasQ Qiskit Fall Fest, 2026. © 2026 Benjamin Tirado. Text, figures and explanations are used under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); code is released under the [Apache License 2.0](https://www.apache.org/licenses/LICENSE-2.0), the same license as Qiskit. Changes: the banner image, table of contents and closing sections of the tutorial were removed (their content appears in Part B), and the level descriptions refer to Part A. The tutorial is built with [Qiskit](https://www.ibm.com/quantum/qiskit), [rustworkx](https://www.rustworkx.org/) and [SciPy](https://scipy.org/), which remain the property of their respective authors.